
# Pipeline overview 

- Load **X** (genes) and **Y** (proteins); pick **one protein** to predict.
- Take a **tiny subset** of samples/genes to keep it fast.
- Do **leave-one-out (LOO)**: for each sample, hold it out as test.
- On the remaining training set, run **inner K-fold CV** to pick XGBoost hyperparams.
- Within each inner fold, do **feature selection**:
  - Compute **p-values** per gene (default: **Pearson**).
  - Keep the **top N%** with lowest p-values (default: **5%**).
- Train XGBoost on the selected features for each param combo; pick the one with **lowest MSE**.
- **Recompute feature selection** on full train with the chosen threshold; **refit** XGBoost and **predict** the held-out sample.
- Collect a row per test sample: **true value**, **predicted value**, **chosen params**, and (if available) **feature importances**.
- Finally, show a quick **RMSE** over the LOO predictions.



In [1]:
# --- Cell 1: Setup (paths, deps, imports) -------------------------------
import os, sys, subprocess, importlib
from pathlib import Path

# Detect project root (works if notebook is at project root or inside it)
CWD = Path.cwd()
if (CWD / "src" / "pipeline_functions.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "src" / "pipeline_functions.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    # last resort: assume repo name
    maybe = [p for p in CWD.parents if (p / "src" / "pipeline_functions.py").exists()]
    PROJECT_ROOT = maybe[0] if maybe else CWD

SRC_DIR  = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path so we can import your code
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Light dep check (install if missing for local notebooks)
def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        pip_name = pip_name or pkg
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pkg, pipn in [
    ("joblib","joblib"),
    ("pandas","pandas"),
    ("numpy","numpy"),
    ("sklearn","scikit-learn"),
    ("xgboost","xgboost>=2.0"),
    ("openpyxl","openpyxl>=3.1"),
]:
    _ensure(pkg, pipn)

import numpy as np, pandas as pd
from joblib import dump, load
import xgboost as xgb

from src.pipeline_functions import cv_pipeline_loo

print("Project root:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


Project root: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb
SRC_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\src
DATA_DIR: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data


In [2]:
# --- Cell 2: Create a fast synthetic dataset and save to data/ ----------
# Small and quick: ~30 samples, 300 features, 3 proteins
rng = np.random.default_rng(123)
n_samples, n_features, n_proteins = 30, 300, 3

X_np = rng.normal(size=(n_samples, n_features))
# make P0 depend on a sparse subset of features + noise; P1 similar; P2 mostly noise
w0 = np.zeros(n_features); w0[:15] = rng.normal(0, 1, size=15)
w1 = np.zeros(n_features); w1[50:70] = rng.normal(0, 1, size=20)

y0 = X_np @ w0 + rng.normal(0, 0.5, size=n_samples)
y1 = X_np @ w1 + rng.normal(0, 0.6, size=n_samples)
y2 = rng.normal(0, 1.0, size=n_samples)

samples  = [f"S{i:02d}" for i in range(n_samples)]
features = [f"G{j:04d}" for j in range(n_features)]
proteins = [f"P{k}" for k in range(n_proteins)]

X = pd.DataFrame(X_np, index=samples, columns=features)
Y = pd.DataFrame({"P0": y0, "P1": y1, "P2": y2}, index=samples)

# Save exactly with the filenames your pipeline expects
x_path = DATA_DIR / "transcriptomics_filtered.joblib"
y_path = DATA_DIR / "proteomics_matched.joblib"
dump(X, x_path); dump(Y, y_path)

display(X.shape, Y.shape, x_path, y_path)


(30, 300)

(30, 3)

WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/pipeline_xgb/data/transcriptomics_filtered.joblib')

WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/pipeline_xgb/data/proteomics_matched.joblib')

In [3]:
# --- Cell 3: Direct function smoke test (very small grid; CPU hist) -----
param_grid = {
    "max_depth": [3],
    "learning_rate": [0.1],
    "n_estimators": [200],          # small for speed; early stopping will help too
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_alpha": [0.0, 0.1],        # 2 combos total
    "reg_lambda": [1.0],
    "tree_method": ["hist"],        # keep CPU to avoid GPU requirement in notebooks
    "max_bin": [256],
}

protein_name = "P0"
res_df = cv_pipeline_loo(
    model=xgb.XGBRegressor(objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0),
    param_grid=param_grid,
    X=X,
    y=Y,
    protein_name=protein_name,
    feature_selector="spearman",  # "pearson" / "spearman" / "anova"
    n_splits=2,                   # inner CV folds (tiny for speed)
    select_k=50,                  # top-k by p-value inside folds
    preselect_k=100,              # quick corr cap
    early_stopping_rounds=10,
)

display(res_df.head())
print("Rows (should equal #samples):", len(res_df))

# Quick metrics on the LOO predictions
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(res_df["true"], res_df["predicted"])
from scipy.stats import pearsonr
r, p = pearsonr(res_df["true"], res_df["predicted"])
print(f"MSE: {mse:.4f} | Pearson r: {r:.3f} (p={p:.1e})")
assert res_df["predicted"].notna().all(), "NaNs in predictions!"
print("✅ Function pipeline smoke test passed.")


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,features_selected
0,P0,S00,-0.082451,-0.092046,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'G0005': 0.013025827705860138, 'G0010': 0.048...","[G0005, G0010, G0253, G0127, G0008, G0121, G01..."
1,P0,S01,-0.257422,-2.633395,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'G0005': 0.004779870621860027, 'G0010': 0.047...","[G0005, G0010, G0127, G0187, G0121, G0253, G00..."
2,P0,S02,-0.611598,-0.049049,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'G0005': 0.006365352310240269, 'G0010': 0.022...","[G0005, G0010, G0127, G0014, G0121, G0253, G02..."
3,P0,S03,-6.591260,1.025449,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'G0005': 0.015533688478171825, 'G0127': 0.012...","[G0005, G0127, G0014, G0253, G0010, G0083, G01..."
4,P0,S04,-4.048434,-1.676336,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'G0005': 0.006113804876804352, 'G0010': 0.020...","[G0005, G0010, G0019, G0127, G0272, G0008, G02..."


Rows (should equal #samples): 30
MSE: 18.7532 | Pearson r: -0.440 (p=1.5e-02)
✅ Function pipeline smoke test passed.


In [5]:
# --- Cell 4: CLI end-to-end test (runs src/pipeline.py) -----------------
import sys, json, time
cli = [
    sys.executable,               # python executable for this kernel
    str(SRC_DIR / "pipeline.py"),
    "--protein", "P1",            # try another protein
    "--pipeline", "single",       # alias for 'loo'
    "--model", "xgboost",
    "--splits", "2",
    "--feature_selector", "pearson",
    "--select_k", "40",
    "--preselect_k", "80",
    "--early_stopping", "10",
    # no --force_gpu so it can run on CPU
]

print("Running:", " ".join(cli))
ret = subprocess.run(cli, cwd=str(SRC_DIR), capture_output=True, text=True)
print("Return code:", ret.returncode)
print("--- STDOUT ---")
print(ret.stdout[-2000:])
print("--- STDERR ---")
print(ret.stderr[-2000:])

assert ret.returncode == 0, "CLI run failed"

# Load the results saved by pipeline.py
out_path = DATA_DIR / "P1_loo_xgboost_LOO_results.joblib"
assert out_path.exists(), f"Expected results file not found: {out_path}"
cli_df = load(out_path)
display(cli_df.head(), out_path)

mse = mean_squared_error(cli_df["true"], cli_df["predicted"])
r, p = pearsonr(cli_df["true"], cli_df["predicted"])
print(f"[CLI] MSE: {mse:.4f} | Pearson r: {r:.3f} (p={p:.1e})")
print("✅ CLI pipeline smoke test passed.")


Running: c:\Users\hp\anaconda3\python.exe c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\src\pipeline.py --protein P1 --pipeline single --model xgboost --splits 2 --feature_selector pearson --select_k 40 --preselect_k 80 --early_stopping 10
Return code: 0
--- STDOUT ---
Loading data...
Running loo for protein 'P1' (LOO across samples)...
Saving: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data\P1_loo_xgboost_LOO_results.joblib
Done.

--- STDERR ---



,protein,LOO sample,true,predicted,params_dict,features_coef_dict,features_selected
0,P1,S00,1.011079,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,"[G0068, G0154, G0182, G0219, G0014, G0059, G01..."
1,P1,S01,0.951921,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,"[G0068, G0154, G0219, G0182, G0014, G0059, G02..."
2,P1,S02,3.047908,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,"[G0068, G0154, G0182, G0201, G0014, G0219, G00..."
3,P1,S03,4.248288,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,"[G0068, G0154, G0182, G0219, G0014, G0016, G00..."
4,P1,S04,-0.533247,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,"[G0068, G0154, G0182, G0014, G0219, G0059, G02..."


WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/pipeline_xgb/data/P1_loo_xgboost_LOO_results.joblib')

ValueError: Input contains NaN.

In [ ]:
# --- Cell 5 (optional): visualize predictions vs truth ------------------
import matplotlib.pyplot as plt

def quick_scatter(df, title):
    plt.figure(figsize=(4.5,4.5))
    plt.scatter(df["true"], df["predicted"])
    lims = [min(df["true"].min(), df["predicted"].min()),
            max(df["true"].max(), df["predicted"].max())]
    plt.plot(lims, lims)
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    plt.show()

quick_scatter(res_df, "Function LOO: P0")
quick_scatter(cli_df, "CLI LOO: P1")


⚠️ GPU probe failed. That's okay on CPU-only machines. Error:
[17:55:15] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\tree\updater_gpu_hist.cu:861: Exception in gpu_hist: [17:55:15] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\tree\updater_gpu_hist.cu:867: Check failed: ctx_->Ordinal() >= 0 (-1 vs. 0) : Must have at least one device

